In [ ]:
#| default_exp apisurface

In [ ]:
#| export
import keyword,inspect
from inspect import Parameter, Signature
from urllib.parse import urlparse, urljoin
from fastcore.imports import *
from fastcore.basics import *
from fastcore.xtras import UNSET
from fastcore.docments import ann_parts

In [ ]:
#| hide
from collections import namedtuple
from types import SimpleNamespace
from nbdev.showdoc import *
from fastcore.test import *

# API surface

> Turn operation metadata into documented, introspectable callables: real signatures, informative docstrings, and browsable grouped namespaces

#| export
A client generated from a machine-readable spec (an OpenAPI document, a Google Discovery document, an SDK symbol graph) is only pleasant to use if its runtime-created callables behave like hand-written ones: tab completion shows real parameters, `?` and `doc()` show real docs, and related operations sit together in namespaces you can browse. This module builds those pieces from plain *operation records*, ducks with these attributes: `name`, `group` (a nesting path: string or list), `summary`, `docs_url`, `params` (ordered parameter names), `required_params`, `param_types`, `param_defaults`, and `param_docs`. `mk_sig` turns a record into an `inspect.Signature` (spec names sanitized to Python identifiers via `sanitized_params`), `mk_doc` renders a docstring, and `OpGroup`/`mk_groups`/`full_docs` assemble named callables into an attribute-chained tree with markdown summaries at every level.

`fastspec` builds its HTTP clients on this layer.

## Names

In [ ]:
#| export
_pat_non_alnum = re.compile(r"[^a-zA-Z0-9]+")

def snake(s: str):
    "Convert an identifier-ish string to snake_case."
    s = _pat_non_alnum.sub("_", s).strip("_")
    s = re.sub(r"(.)([A-Z][a-z]+)", r"\1_\2", s)
    s = re.sub(r"([a-z0-9])([A-Z])", r"\1_\2", s)
    return s.lower().strip("_")

Spec names arrive in every convention at once: OpenAPI operation ids like `tunedModels.generateContent`, header-ish parameter names like `max-tokens`, Swift argument labels like `networkAccessAllowed`. `snake` normalizes any of them to one Python style. It differs from `camel2snake` by also folding runs of non-alphanumeric characters into underscores, so dotted and hyphenated names come out as identifiers too.

In [ ]:
test_eq(snake('tunedModels.generateContent'), 'tuned_models_generate_content')
test_eq(snake('max-tokens'), 'max_tokens')
test_eq(snake('networkAccessAllowed'), 'network_access_allowed')
snake('HTMLParser v2')

'html_parser_v2'

`sanitize_param_name` is the stricter form for parameter names, which must be assignable: anything non-identifier becomes `_` before snaking.

In [ ]:
#| export
def sanitize_param_name(p): return snake(re.sub(r'\W', '_', p).strip('_'))

## Signatures

The running example throughout: one operation record for a chat-message API, with a hyphenated name and a Python keyword among its parameters. Any object with these attributes works.

In [ ]:
send = SimpleNamespace(
    group='messages', name='send', summary='Send a chat message',
    docs_url='https://api.example.com/docs#send',
    params=['model', 'input', 'max-tokens', 'for', 'stream'],
    required_params=['model', 'input'],
    param_types=dict(model=str, input=str, stream=bool, **{'max-tokens':int, 'for':str}),
    param_defaults=dict(stream=False),
    param_docs={'model':'Model id', 'input':'Prompt text', 'max-tokens':'Cap on generated tokens', 'for':'End-user id'})

In [ ]:
#| export
def sanitized_params(ps):
    "Mapping from spec param names `ps` to valid Python identifiers; exact names win collisions."
    def _sani(p):
        name = sanitize_param_name(p)
        if keyword.iskeyword(name): name += '_'
        return name
    res,used = {},{p for p in ps if _sani(p)==p}
    for p in ps:
        name = _sani(p)
        if name!=p:
            while name in used: name += '_'
        used.add(name)
        res[p] = name
    return res

Two of those parameter names can't be Python parameters as-is: `max-tokens` isn't an identifier, and `for` is a keyword. `sanitized_params` maps every spec name to a usable identifier, leaving good names alone; a name that's already exact always keeps its spelling, and invented names grow underscores until they're unique.

In [ ]:
sparams = sanitized_params(send.params)
test_eq(sparams['max-tokens'], 'max_tokens')
test_eq(sparams['for'], 'for_')
sparams

{'model': 'model',
 'input': 'input',
 'max-tokens': 'max_tokens',
 'for': 'for_',
 'stream': 'stream'}

In [ ]:
#| export
def _mk_param(name, required, anno=None, default=Parameter.empty):
    "Create a function signature parameter."
    anno = Parameter.empty if anno is None else anno
    if default is Parameter.empty: default = Parameter.empty if required else UNSET
    return Parameter(name, kind=Parameter.POSITIONAL_OR_KEYWORD, default=default, annotation=anno)

def _sort_key(o):
    if o.default is Parameter.empty: return 0
    if o.default is UNSET: return 1
    return 2

def mk_sig(op, sparams=None, defaults=None):
    "An operation signature with parameter descriptions in `Annotated`; `defaults` makes those params optional"
    if sparams is None: sparams = sanitized_params(op.params)
    defaults = defaults or {}
    params = []
    for pname, sname in sparams.items():
        default = defaults.get(pname, op.param_defaults.get(pname, Parameter.empty))
        anno = op.param_types.get(pname)
        if doc := op.param_docs.get(pname): anno = typing.Annotated[anno or typing.Any, doc]
        params.append(_mk_param(sname, pname in op.required_params, anno, default))
    return Signature(sorted(params, key=_sort_key))

`mk_sig` builds the real `inspect.Signature`: required parameters first, then optional ones. An optional parameter without a declared default uses `UNSET` (omit it from the request), not `None` (which sends JSON `null`). Descriptions travel with parameter types as `Annotated` metadata. `docments` can read them without source code, including after `delegates` copies the parameters to a wrapper.

In [ ]:
sig = mk_sig(send)
test_eq(typing.get_args(sig.parameters['model'].annotation), (str, 'Model id'))
sig

<Signature (model: Annotated[str, 'Model id'], input: Annotated[str, 'Prompt text'], max_tokens: Annotated[int, 'Cap on generated tokens'] = UNSET, for_: Annotated[str, 'End-user id'] = UNSET, stream: bool = False)>

Passing `defaults` makes those parameters optional with the given values. This supports client-level binding without changing the spec. An explicit `None` remains distinct from `UNSET`.

In [ ]:
bound = mk_sig(send, defaults={'model':'sonnet-4', 'for':None})
test_eq(bound.parameters['model'].default, 'sonnet-4')
test_is(bound.parameters['for_'].default, None)
test_is(mk_sig(send, defaults={'model':None}).parameters['model'].default, None)
bound

<Signature (input: Annotated[str, 'Prompt text'], max_tokens: Annotated[int, 'Cap on generated tokens'] = UNSET, model: Annotated[str, 'Model id'] = 'sonnet-4', for_: Annotated[str, 'End-user id'] = None, stream: bool = False)>

## Docstrings

In [ ]:
#| export
def _op_summary(op):
    'Single line op summary with fallback to `op.name` and relative link rewriting'
    s = re.sub(r"\s+", " ", str(op.summary or op.name)).strip() or str(op.name)
    if not op.docs_url: return s
    p = urlparse(op.docs_url)
    base = f"{p.scheme}://{p.netloc}"
    return re.sub(r"\]\((/[^)]+)\)", lambda m: f"]({urljoin(base, m[1].strip())})", s)

`_op_summary` uses the operation name when a summary is absent. Whitespace in spec summaries collapses to one space.

In [ ]:
TestOp = namedtuple('TestOp', 'summary name docs_url')
test_eq(_op_summary(TestOp("List models", "list", "")), "List models")
test_eq(_op_summary(TestOp("", "list_models", "")), "list_models")
test_eq(_op_summary(TestOp(None, "list_models", "")), "list_models")

test_eq(_op_summary(TestOp("List  all\n models", "list", "")), "List all models")

Root-relative links resolve against `docs_url`. Anchor-only links and absolute URLs remain unchanged.

In [ ]:
test_eq(_op_summary(TestOp("[details](/docs#rate)", "x", "https://api.example.com/docs")),
    "[details](https://api.example.com/docs#rate)")
test_eq(_op_summary(TestOp("[see](#limits)", "x", "https://api.example.com/docs")), "[see](#limits)")
test_eq(_op_summary(TestOp("[info](https://other.com/x)", "x", "https://api.example.com/docs")),
    "[info](https://other.com/x)")

In [ ]:
#| export
def mk_doc(op, sig, sparams):
    "Render operation docstring with summary, docs URL, and parameter hints."
    lines = [_op_summary(op)]
    if op.docs_url: lines.append(f"\nDocs: {op.docs_url}")
    if sig.parameters:
        lines.append("\nParameters:")
        req = set(op.required_params or [])
        # Reverse map: sanitized → original, for looking up docs/required
        rsparams = {v:k for k,v in sparams.items()}
        for nm,p in sig.parameters.items():
            orig = rsparams.get(nm, nm)
            r = f"default: {p.default!r}" if p.default not in (Parameter.empty, UNSET) else "required" if orig in req else "optional"
            ann = '' if p.annotation is Parameter.empty else inspect.formatannotation(ann_parts(p.annotation)[0]).removeprefix('typing.')
            desc = (op.param_docs or {}).get(orig, "")
            lines.append(f"- {nm} ({ann}, {r}){': ' + desc if desc else ''}")
    return "\n".join(lines)

`mk_doc` renders the docstring a generated callable should carry: summary, docs link, and one line per parameter with its type, requiredness or default, and spec description — under the *sanitized* name the caller will actually type.

In [ ]:
send.__signature__ = sig
send.__doc__ = mk_doc(send, sig, sparams)
PrettyString(send.__doc__)

Send a chat message

Docs: https://api.example.com/docs#send

Parameters:
- model (str, required): Model id
- input (str, required): Prompt text
- max_tokens (int, optional): Cap on generated tokens
- for_ (str, optional): End-user id
- stream (bool, default: False)

## Groups

In [ ]:
#| export
def _op_line(op, sig):
    head = f"{'.'.join(snake(g) for g in listify(op.group))}.{op.name}"
    if op.docs_url: head = f"[{head}]({op.docs_url})"
    s = f"({', '.join(sig.parameters)})"
    summ = _op_summary(op)
    return f"{head}{s}: *{summ}*"

In [ ]:
#| export
class OpGroup:
    "Namespace for grouped operations: each op is an attribute, and the repr lists them all"
    def __init__(self, name: str, ops):
        self.name,self.ops,self.subgroups = name,list(ops),{}
        for op in self.ops: setattr(self, op.name, op)

    @property
    def __doc__(self):
        res = [f"- {_op_line(op, op.__signature__)}" for op in self.ops if hasattr(op, '__signature__')]
        res.extend(f"- {k}/" for k in sorted(self.subgroups))
        return "\n".join(res)

    def __dir__(self): return object.__dir__(self)
    def __allow__(self): return self.ops + list(self.subgroups.values())

    def _repr_markdown_(self): return self.__doc__ + '\n\nOverview only. Read `doc(group.operation)` for parameter details and `doc(group.subgroup)` to descend.'
    __repr__ = _repr_markdown_

`OpGroup` exposes each operation as an attribute. Its overview lists operations with signatures; each operation retains its own parameter documentation.

In [ ]:
messages = OpGroup('messages', [send])
assert 'messages.send' in messages.__doc__
messages

- [messages.send](https://api.example.com/docs#send)(model, input, max_tokens, for_, stream): *Send a chat message*

Overview only. Read `doc(group.operation)` for parameter details and `doc(group.subgroup)` to descend.

In [ ]:
#| export
def mk_groups(ops):
    "Nested tree of `OpGroup`s from ops, following each op's `group` path."
    root = {}
    for op in ops:
        g = [snake(p) for p in listify(op.group)]
        node = root
        for part in g[:-1]: node = node.setdefault(part, {})
        node.setdefault(g[-1], {}).setdefault('_ops', []).append(op)

    def _mk(name, d):
        grp = OpGroup(name, d.pop('_ops', []))
        for k,v in d.items(): setattr(grp, k, grp.subgroups.setdefault(k, _mk(k, v)))
        return grp

    return {k: _mk(k, v) for k,v in root.items()}

`mk_groups` builds a tree from each operation's `group`: a string for a flat namespace, a list for nesting. Here a second copy of `send` belongs to the `messages.batches` subgroup.

In [ ]:
batch_send = SimpleNamespace(**vars(send))
batch_send.group = ['messages', 'batches']
groups = mk_groups([send, batch_send])
test_is(groups['messages'].batches.send, batch_send)
groups['messages']


- [messages.send](https://api.example.com/docs#send)(model, input, max_tokens, for_, stream): *Send a chat message*
- batches/

Overview only. Read `doc(group.operation)` for parameter details and `doc(group.subgroup)` to descend.

Attribute access descends through subgroups to the operation, which retains its parameter documentation:

In [ ]:
PrettyString(groups['messages'].batches.send.__doc__)

Send a chat message

Docs: https://api.example.com/docs#send

Parameters:
- model (str, required): Model id
- input (str, required): Prompt text
- max_tokens (int, optional): Cap on generated tokens
- for_ (str, optional): End-user id
- stream (bool, default: False)

`OpGroup.__allow__` supports hosts whose `allow` mechanism registers callable surfaces recursively: allowing a group allows its ops and every nested subgroup in one call.

## Reference docs

In [ ]:
#| export
def _group_docs(name, g, lvl=2):
    res = [f"{'#'*lvl} {name}"]
    if g.__doc__: res.append(g.__doc__)
    res += [_group_docs(f"{name}.{k}", sub, min(lvl+1, 6)) for k,sub in sorted(g.subgroups.items())]
    return "\n\n".join(res)

def full_docs(groups):
    "Markdown overview of every group and operation in a `mk_groups` tree."
    return "\n\n".join(_group_docs(nm, g) for nm,g in sorted(groups.items()))


`full_docs` combines the group overviews into one Markdown document, including nested groups. It lists operation names, parameter names, and summaries. Parameter types, defaults, and descriptions remain in each operation's documentation.

In [ ]:
PrettyString(full_docs(groups))

## messages

- [messages.send](https://api.example.com/docs#send)(model, input, max_tokens, for_, stream): *Send a chat message*
- batches/

### messages.batches

- [messages.batches.send](https://api.example.com/docs#send)(model, input, max_tokens, for_, stream): *Send a chat message*

## export -

In [ ]:
#| hide
import nbdev
nbdev.nbdev_export()